In [ ]:
import os
import torch
import numpy as np
os.environ["KERAS_BACKEND"] = "torch"

In [ ]:
import keras
import tensorflow as tf

In [ ]:
if keras.backend.backend() != "torch":
    print(f"warning: keras backend is set to {keras.backend.backend()}, restart jupyter kernel!!!!")
    raise RuntimeError()

In [ ]:
import json

global config
with open('./config/keras_nn.json') as keras_nn_config:
    config = json.load(keras_nn_config)
    print("config loaded")

In [ ]:
# Root-level fields
batch_size = config["batchSize"]
scaler_enabled = config["scaler"]["enabled"]
scaler_type = config["scaler"]["type"]

seed = config["seed"]

validation_enabled = config["validation"]["enabled"]
validation_ratio = config["validation"]["ratio"]


In [ ]:
keras.utils.set_random_seed(seed)

In [ ]:
%load_ext tensorboard
# now available at http://localhost:6006/?

In [ ]:
# Dataset initialization

from utils.data_loader import get_ml_cup_data
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

def _scaler():
    if not scaler_enabled:
        return None

    match scaler_type:
        case "Standard":
            return StandardScaler()
        case "MinMax":
            return MinMaxScaler()
        case "Robust":
            return RobustScaler()
        case "MaxAbsScaler":
            return MaxAbsScaler()
        case _:
            return None

train_loader, validation_loader, test_loader, input_size, output_size = get_ml_cup_data(
    batch_size, 
    scaler=_scaler(), 
    validation=validation_enabled, 
    validation_ratio=validation_ratio
    )

In [ ]:
train_loader.dataset.X.shape

In [ ]:
validation_loader.dataset.X.shape

In [ ]:
test_loader.dataset.X.shape

In [ ]:
# --- Neural network ("nn") section ---
nn_optimizer = config["nn"]["optimizer"]
nn_epochs = config["nn"]["epochs"]

# Hidden layers
nn_hidden: list[dict] = config["nn"]["hidden"]

In [ ]:
from keras import Sequential
from keras.layers import Input, Dense

In [ ]:
model = Sequential()
model.add(Input(shape=(input_size,)))
for hidden in nn_hidden:
    units: int = hidden["units"]
    activation: str = hidden["activation"]
    model.add(Dense(units, activation=activation))
model.add(Dense(output_size))

In [ ]:
from losses import MeanEuclidianError

mee = MeanEuclidianError(name="mee")

In [ ]:
model.compile(optimizer=nn_optimizer, loss=mee)

In [ ]:
model.summary()

In [ ]:
from keras.callbacks import EarlyStopping, TensorBoard
import datetime

def log_dir(name, append:str=None):
    BASE = f"logs/{name}/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    if append:
        BASE += "_" + append
    return BASE

In [ ]:
# Early stopping
nn_es_enabled = config["nn"]["earlyStopping"]["enabled"]                ## true/false
nn_es_patience = config["nn"]["earlyStopping"]["patience"]              ## int
nn_es_mode = config["nn"]["earlyStopping"]["mode"]                      ## min/max
nn_es_restore_best = config["nn"]["earlyStopping"]["restoreBestWeight"] ## true/false

In [ ]:
tensorboard_cb = TensorBoard(
    log_dir=log_dir("fit"),
    histogram_freq=1,
    write_graph=True,
    write_images=False
)

early_stopping_cb = EarlyStopping(
    mode=nn_es_mode,
    monitor="val_loss",
    patience=nn_es_patience,
    restore_best_weights=nn_es_restore_best,
)

In [ ]:
tensorboard_cb.log_dir = log_dir("fit", nn_optimizer)

callbacks = [tensorboard_cb]
if nn_es_enabled:
    callbacks.append(early_stopping_cb)
    
history = model.fit(train_loader, validation_data=validation_loader, epochs=nn_epochs, callbacks=callbacks)

In [ ]:
%tensorboard --logdir logs/fit

In [ ]:
# evaluate model
results = model.evaluate(test_loader, return_dict=True)
print(results)

In [ ]:
# write logs to a separate folder
writer = tf.summary.create_file_writer(log_dir("eval", nn_optimizer))

with writer.as_default():
    for k, v in results.items():
        tf.summary.scalar(k, v, step=0)

writer.close()

In [ ]:
%tensorboard --logdir logs/eval

In [ ]:
# --- Optuna ("optuna") section ---
optuna_epochs = config["optuna"]["epochs"]

# Suggestions for hidden layers
optuna_hidden_1_range = config["optuna"]["suggestions"]["hidden"][0]["range"]
optuna_hidden_1_activation = config["optuna"]["suggestions"]["hidden"][0]["activation"]

optuna_hidden_2_range = config["optuna"]["suggestions"]["hidden"][1]["range"]
optuna_hidden_2_activation = config["optuna"]["suggestions"]["hidden"][1]["activation"]

# Optuna early stopping
optuna_es_enabled = config["optuna"]["earlyStopping"]["enabled"]
optuna_es_patience = config["optuna"]["earlyStopping"]["patience"]
optuna_es_mode = config["optuna"]["earlyStopping"]["mode"]
optuna_es_restore_best = config["optuna"]["earlyStopping"]["restoreBestWeight"]

In [ ]:
## Optuna
import optuna
import tensorflow as tf

def objective(trial):
    # Suggest hyperparameters
    units1 = trial.suggest_int("units1", 16, 128)
    units2 = trial.suggest_int("units2", 16, 128)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    # Build model
    model = Sequential([
        Input(shape=(input_size,)),
        # Dense layers are fully connected layers
        Dense(units1, activation='relu'),
        Dense(units2, activation='relu'),
        Dense(output_size)
    ])

    optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss=mee
    )

    tensorboard_cb.log_dir = log_dir("fit", f"OPTUNA_TRIAL#{trial.number}")

    pruning_cb = optuna.integration.KerasPruningCallback(trial, "val_loss")
    
    checkpoint_path = f"checkpoints/optuna_trial_{trial.number}.keras"

    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False
    )
    
    # Train model
    history = model.fit(
        train_loader,
        validation_data=validation_loader,
        epochs=optuna_epochs,
        verbose=0,
        callbacks=[tensorboard_cb, pruning_cb, checkpoint_cb]
    )

    val_loss = history.history["val_loss"][-1]
    return val_loss

In [ ]:
from optuna.samplers import TPESampler

sampler = TPESampler(seed=seed)
study = optuna.create_study(sampler=sampler, direction="minimize")
study.optimize(objective, n_trials=30)

print("Best trial:", study.best_trial.params)

In [ ]:
from utils.optuna import delete_pruned_trial_dirs

In [ ]:
delete_pruned_trial_dirs(root_path="logs/fit", study=study)

In [ ]:
study.best_trial

In [ ]:
best_trial = study.best_trial
best_model_path = f"checkpoints/optuna_trial_{best_trial.number}.keras"
best_model = keras.models.load_model(best_model_path)

In [ ]:
test_loss = best_model.evaluate(test_loader)

In [ ]:
writer = tf.summary.create_file_writer(log_dir("eval", f"OPTUNA_TRIAL#{best_trial.number}"))
with writer.as_default():
    tf.summary.scalar("loss", test_loss, step=0)
    writer.flush()

In [ ]:
%tensorboard --logdir logs/eval